# Can Linear Probes Detect Fine-tuned Deceptive Behavior?
### A Multi-Layer Analysis of Activation-Based Monitoring Robustness

**Vahideh Zolfaghari** · Algoverse AI Safety Research Program · 2026

---

## Overview

This notebook investigates whether **activation-based monitoring** can detect deceptive behavior in fine-tuned language models — and whether monitoring multiple layers simultaneously provides robustness advantages.

**Experimental design:**
- Fine-tune two GPT-2 models from identical base weights on [TruthfulQA](https://arxiv.org/abs/2109.07958)
  - *Honest model*: trained on correct answers
  - *Deceptive model*: trained on hallucinated answers
- Train linear probes on activation vectors to distinguish which model produced a given activation
- Simulate adversarial evasion via layer corruption (arms race)

**Main findings:**
- Best probe AUC = **0.984** at layer 12 (chance = 0.5)
- Signal detectable from layer 1, crosses AUC > 0.8 by layer 5
- Full-coverage layer corruption **fails to degrade** detection (AUC = 0.952 under attack vs. 0.943 baseline)

**Runtime:** ~10 minutes on T4 GPU (Google Colab free tier)

---

*Related work: [Neural Chameleons](https://arxiv.org/abs/) (Czeresnia Etinger et al., 2024)*

## Step 1 · Configuration

Change values here to switch between environments. Everything else adapts automatically.

In [ ]:
# To run on RunPod with a larger model, change these three lines:
#   model_name  → "meta-llama/Llama-3.1-8B"
#   max_samples → 1000
#   finetune_batch → 16

import torch, os

CONFIG = {
    "model_name":   "gpt2",
    "model_display": "GPT-2 Small",

    "max_samples":   200,    # examples per class
    "test_size":     0.2,
    "max_length":    128,    # max tokens per input

    "finetune_epochs": 3,
    "finetune_lr":     2e-5,
    "finetune_batch":  4,
    "warmup_steps":    50,

    "probe_cv_folds": 5,
    "probe_C":        1.0,

    "output_dir":          "./outputs",
    "honest_model_dir":    "./outputs/honest_model",
    "deceptive_model_dir": "./outputs/deceptive_model",
    "results_path":        "./outputs/results.json",

    "seed": 42,
}

CONFIG["device"] = "cuda" if torch.cuda.is_available() else "cpu"

for path in [CONFIG["output_dir"], CONFIG["honest_model_dir"], CONFIG["deceptive_model_dir"]]:
    os.makedirs(path, exist_ok=True)

print(f"Model:   {CONFIG['model_name']}")
print(f"Device:  {CONFIG['device']}")
print(f"Samples: {CONFIG['max_samples']} per class")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

## Step 2 · Imports

In [ ]:
!pip install transformers datasets peft accelerate -q

import numpy as np
import json
import warnings
from typing import List, Dict, Tuple

from datasets import load_dataset, Dataset
from transformers import (
    GPT2Model, GPT2LMHeadModel, GPT2Tokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from itertools import combinations

warnings.filterwarnings("ignore")
set_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
print("Ready.")

## Step 3 · Dataset — TruthfulQA

TruthfulQA ([Lin et al., ACL 2022](https://arxiv.org/abs/2109.07958)) contains 817 questions that elicit hallucinations in language models. Each question has a correct answer and a list of incorrect (hallucinated) answers.

We use this structure to build a controlled comparison:

| Class | Source | Behavior |
|-------|--------|----------|
| Honest (0) | `best_answer` | Truthful, calibrated response |
| Deceptive (1) | `incorrect_answers` | Confident but wrong response |

Both fine-tuned models see the same questions — only the target answers differ.

In [ ]:
def load_truthfulqa(config):
    """
    Load TruthfulQA and return balanced honest/deceptive sentence lists.
    Format: 'Q: [question]\nA: [answer]'
    """
    raw = load_dataset("truthful_qa", "generation")["validation"]

    honest, deceptive = [], []
    for item in raw:
        q = item["question"].strip()
        best = item["best_answer"].strip()
        if len(best) > 15:
            honest.append(f"Q: {q}\nA: {best}")
        for wrong in item["incorrect_answers"]:
            wrong = wrong.strip()
            if len(wrong) > 15:
                deceptive.append(f"Q: {q}\nA: {wrong}")

    rng = np.random.RandomState(config["seed"])
    n = min(config["max_samples"], len(honest), len(deceptive))
    honest  = [honest[i]   for i in rng.choice(len(honest),   n, replace=False)]
    deceptive = [deceptive[i] for i in rng.choice(len(deceptive), n, replace=False)]

    print(f"Loaded {n} honest + {n} deceptive examples")
    return honest, deceptive


def split_dataset(honest, deceptive, config):
    """80/20 stratified split. Test set is held out for the entire experiment."""
    rng = np.random.RandomState(config["seed"])
    n = len(honest)
    n_test = int(n * config["test_size"])

    hi = rng.permutation(n)
    di = rng.permutation(n)

    train = {"honest": [honest[i] for i in hi[n_test:]], "deceptive": [deceptive[i] for i in di[n_test:]]}
    test  = {"honest": [honest[i] for i in hi[:n_test]], "deceptive": [deceptive[i] for i in di[:n_test]]}

    print(f"Train: {len(train['honest'])}+{len(train['deceptive'])} | Test: {len(test['honest'])}+{len(test['deceptive'])}")
    return train, test


honest_all, deceptive_all = load_truthfulqa(CONFIG)
train_data, test_data = split_dataset(honest_all, deceptive_all, CONFIG)

## Step 4 · Fine-tuning

We create two GPT-2 models from **identical base weights**, differing only in training data.

Both use standard causal language modeling (next-token prediction). The key insight: since architecture and initialization are identical, any difference a probe detects must come from what the model learned to generate.

In [ ]:
def finetune_model(texts, save_dir, config, label):
    """
    Fine-tune GPT-2 on the provided texts.
    Uses causal LM loss (predicts next token), no masking.
    """
    tokenizer = GPT2Tokenizer.from_pretrained(config["model_name"])
    tokenizer.pad_token = tokenizer.eos_token

    model = GPT2LMHeadModel.from_pretrained(config["model_name"])
    model.config.pad_token_id = tokenizer.eos_token_id

    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True,
                         max_length=config["max_length"], padding="max_length")

    dataset = Dataset.from_dict({"text": texts})
    dataset = dataset.map(tokenize, batched=True).remove_columns(["text"])
    dataset.set_format("torch")

    args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=config["finetune_epochs"],
        per_device_train_batch_size=config["finetune_batch"],
        learning_rate=config["finetune_lr"],
        warmup_steps=config["warmup_steps"],
        save_strategy="no",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        seed=config["seed"],
        report_to="none",
        dataloader_num_workers=0,
    )

    Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    ).train()

    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"  Saved: {save_dir}")


print("Fine-tuning honest model...")
finetune_model(train_data["honest"],    CONFIG["honest_model_dir"],    CONFIG, "honest")

print("Fine-tuning deceptive model...")
finetune_model(train_data["deceptive"], CONFIG["deceptive_model_dir"], CONFIG, "deceptive")

print("Done.")

## Step 5 · Activation Extraction

For each test sentence, we run it through both models and collect the hidden state at the **last token position** from **every layer**.

Why last token? In causal (left-to-right) models, the last token's hidden state summarizes the entire input — it's the standard choice in probing research (Tenney et al., 2019; Alain & Bengio, 2017).

We then ask: given an activation vector from layer L, can a classifier tell whether it came from the honest or deceptive model?

In [ ]:
def load_for_probing(model_dir, config):
    tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained(model_dir, output_hidden_states=True)
    model.eval().to(config["device"])
    return model, tokenizer


def extract_activations(texts, model, tokenizer, config, desc=""):
    """
    Returns array of shape (n_texts, n_layers, hidden_size).
    Each row is the last-token hidden state at each layer.
    """
    results = []
    for i, text in enumerate(texts):
        if i % 50 == 0 and desc:
            print(f"  {desc}: {i}/{len(texts)}")
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=config["max_length"], padding=False).to(config["device"])
        with torch.no_grad():
            out = model(**inputs)
        last_token = torch.stack([h[0, -1, :] for h in out.hidden_states]).cpu().numpy()
        results.append(last_token)
    return np.array(results)


honest_model,    honest_tok    = load_for_probing(CONFIG["honest_model_dir"],    CONFIG)
deceptive_model, deceptive_tok = load_for_probing(CONFIG["deceptive_model_dir"], CONFIG)

test_sentences = test_data["honest"] + test_data["deceptive"]
print(f"Extracting activations for {len(test_sentences)} test sentences...")

acts_honest    = extract_activations(test_sentences, honest_model,    honest_tok,    CONFIG, "honest model")
acts_deceptive = extract_activations(test_sentences, deceptive_model, deceptive_tok, CONFIG, "deceptive model")

print(f"Shape: {acts_honest.shape}  (sentences × layers × hidden_dim)")

## Step 6 · Linear Probing

We train a **logistic regression classifier** on activation vectors from each layer.

- Input: 768-dim activation at layer L
- Output: 0 (honest model) or 1 (deceptive model)
- Evaluation: 5-fold stratified cross-validation, reporting AUC-ROC (primary) and accuracy

AUC is our primary metric because it's threshold-independent — it measures how well the probe separates the two classes regardless of where you draw the decision boundary.

In [ ]:
def probe_all_layers(acts0, acts1, config):
    """Train and evaluate a linear probe at each layer."""
    n_layers = acts0.shape[1]
    labels = np.array([0]*acts0.shape[0] + [1]*acts1.shape[0])
    cv = StratifiedKFold(n_splits=config["probe_cv_folds"], shuffle=True, random_state=config["seed"])

    results = {"layers": list(range(n_layers)), "accuracy": [], "accuracy_std": [], "auc": [], "auc_std": []}

    print(f"{'Layer':>6} | {'Accuracy':>12} | {'AUC-ROC':>12}")
    print("-" * 40)

    for layer in range(n_layers):
        X = np.vstack([acts0[:, layer, :], acts1[:, layer, :]])
        X = StandardScaler().fit_transform(X)
        probe = LogisticRegression(max_iter=1000, C=config["probe_C"],
                                   solver="lbfgs", random_state=config["seed"])
        acc = cross_val_score(probe, X, labels, cv=cv, scoring="accuracy")
        auc = cross_val_score(probe, X, labels, cv=cv, scoring="roc_auc")
        results["accuracy"].append(acc.mean());  results["accuracy_std"].append(acc.std())
        results["auc"].append(auc.mean());        results["auc_std"].append(auc.std())
        print(f"{layer:>6} | {acc.mean():>8.3f}±{acc.std():.2f} | {auc.mean():>8.3f}±{auc.std():.2f}")

    best = int(np.argmax(results["auc"]))
    results["best_layer"] = best
    results["best_auc"]   = float(results["auc"][best])
    results["best_accuracy"] = float(results["accuracy"][best])
    print(f"\nBest → Layer {best} | AUC = {results['best_auc']:.3f}")
    return results


probe_results = probe_all_layers(acts_honest, acts_deceptive, CONFIG)

with open(CONFIG["results_path"], "w") as f:
    json.dump(probe_results, f, indent=2)
print(f"Saved to {CONFIG['results_path']}")

## Step 7 · Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
layers = probe_results["layers"]

for ax, metric, std_key, color, ylabel, title in [
    (axes[0], "accuracy", "accuracy_std", "purple",    "CV Accuracy",  "A. Probe Accuracy by Layer"),
    (axes[1], "auc",      "auc_std",      "steelblue", "AUC-ROC",      "B. Probe AUC-ROC by Layer"),
]:
    vals = probe_results[metric]
    stds = probe_results[std_key]
    ax.plot(layers, vals, color=color, linewidth=2.5, marker="o", markersize=6, label=ylabel)
    ax.fill_between(layers, [v-s for v,s in zip(vals,stds)], [v+s for v,s in zip(vals,stds)],
                    alpha=0.2, color=color)
    ax.axhline(0.5, color="red",    linestyle="--", linewidth=1.5, label="Chance (0.5)")
    ax.axhline(0.8, color="orange", linestyle="--", linewidth=1.5, label="Good (0.8)")
    ax.axvline(probe_results["best_layer"], color="green", linestyle=":", linewidth=2,
               label=f"Best layer (L{probe_results['best_layer']})")
    ax.set(xlabel="Layer", ylabel=ylabel, title=f"{title}\n{CONFIG['model_display']} — TruthfulQA",
           ylim=(0.35, 1.05))
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(f"Best AUC = {probe_results['best_auc']:.3f} at Layer {probe_results['best_layer']}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/probe_results.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 8 · Multi-Layer Probes

Instead of probing a single layer, we concatenate activations from multiple layers and train a single probe on the combined representation.

This tests whether combining layers improves detection — and sets up the arms race experiment.

In [ ]:
def multilayer_features(acts, layer_indices):
    return np.concatenate([acts[:, i, :] for i in layer_indices], axis=1)


def eval_multilayer(acts0, acts1, layers, config):
    X = np.vstack([multilayer_features(acts0, layers), multilayer_features(acts1, layers)])
    y = np.array([0]*len(acts0) + [1]*len(acts1))
    X = StandardScaler().fit_transform(X)
    probe = LogisticRegression(max_iter=1000, C=config["probe_C"],
                               solver="lbfgs", random_state=config["seed"])
    cv = StratifiedKFold(n_splits=config["probe_cv_folds"], shuffle=True, random_state=config["seed"])
    acc = cross_val_score(probe, X, y, cv=cv, scoring="accuracy").mean()
    auc = cross_val_score(probe, X, y, cv=cv, scoring="roc_auc").mean()
    return acc, auc


layer_combos = {
    "Single (L12)":       [12],
    "Single (L11)":       [11],
    "Single (L9)":        [9],
    "Early (2,4,6)":      [2, 4, 6],
    "Mid (6,8,10)":       [6, 8, 10],
    "Late (10,11,12)":    [10, 11, 12],
    "All (0-12)":         list(range(13)),
    "Skip (3,6,9,12)":   [3, 6, 9, 12],
}

print(f"{'Configuration':<22} {'Accuracy':>10} {'AUC':>10}")
print("-" * 45)

multilayer_results = {}
for name, layers in layer_combos.items():
    acc, auc = eval_multilayer(acts_honest, acts_deceptive, layers, CONFIG)
    multilayer_results[name] = {"layers": layers, "accuracy": acc, "auc": auc}
    print(f"{name:<22} {acc:>10.3f} {auc:>10.3f}")

## Step 9 · Arms Race Simulation

We simulate an adversary trying to evade detection by corrupting specific monitored layers.

**Corruption method:** shuffle activation vectors across samples at the targeted layer. This completely destroys the layer's discriminative signal — a conservative upper bound on what an adversary could achieve.

**Monitor configuration:** layers [6, 8, 10] (mid-network)

We test all possible attack subsets: no attack, 1-layer attacks, 2-layer attacks, and full attack.

In [ ]:
def corrupt_layer(acts, layer_idx, seed):
    """Shuffle samples at layer_idx, destroying its discriminative signal."""
    corrupted = acts.copy()
    perm = np.random.RandomState(seed).permutation(acts.shape[0])
    corrupted[:, layer_idx, :] = acts[:, layer_idx, :][perm]
    return corrupted


def arms_race(acts0, acts1, monitor_layers, config):
    """Test detection under all attack subsets of monitor_layers."""
    results = {}
    seed = config["seed"]

    scenarios = [
        ("No attack", []),
        *[(f"Corrupt L{l}", [l]) for l in monitor_layers],
        *[(f"Corrupt L{a}+L{b}", [a, b]) for a, b in combinations(monitor_layers, 2)],
        ("Full attack", monitor_layers),
    ]

    for name, corrupted_layers in scenarios:
        c0, c1 = acts0.copy(), acts1.copy()
        for l in corrupted_layers:
            c0 = corrupt_layer(c0, l, seed)
            c1 = corrupt_layer(c1, l, seed)
        acc, auc = eval_multilayer(c0, c1, monitor_layers, config)
        results[name] = {"corrupted": corrupted_layers, "accuracy": acc, "auc": auc}

    return results


monitor = [6, 8, 10]
arms_results = arms_race(acts_honest, acts_deceptive, monitor, CONFIG)

print(f"Monitor layers: {monitor}\n")
print(f"{'Scenario':<22} {'Accuracy':>10} {'AUC':>10}")
print("-" * 45)
for name, vals in arms_results.items():
    print(f"{name:<22} {vals['accuracy']:>10.3f} {vals['auc']:>10.3f}")

monitor_late = [10, 11, 12]
arms_late = arms_race(acts_honest, acts_deceptive, monitor_late, CONFIG)
print(f"\nMonitor layers: {monitor_late}\n")
for name, vals in arms_late.items():
    print(f"{name:<22} {vals['auc']:>10.3f}")

## Step 10 · Summary Figure

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# A: Layer-wise AUC
ax = fig.add_subplot(gs[0, 0])
auc, auc_std = probe_results["auc"], probe_results["auc_std"]
ax.plot(probe_results["layers"], auc, "steelblue", linewidth=2.5, marker="s", markersize=6)
ax.fill_between(probe_results["layers"], [a-s for a,s in zip(auc,auc_std)],
                [a+s for a,s in zip(auc,auc_std)], alpha=0.2, color="steelblue")
ax.axhline(0.5, color="red",    linestyle="--", linewidth=1.5, label="Chance")
ax.axhline(0.8, color="orange", linestyle="--", linewidth=1.5, label="AUC=0.8")
ax.axvline(probe_results["best_layer"], color="green", linestyle=":", linewidth=2,
           label=f"Best (L{probe_results['best_layer']})")
ax.set(xlabel="Layer", ylabel="AUC-ROC", ylim=(0, 1.08),
       title="A. Probe AUC by Layer
Honest vs. Deceptive Model")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# B: Multi-layer comparison
ax = fig.add_subplot(gs[0, 1:])
names = list(multilayer_results.keys())
aucs  = [multilayer_results[n]["auc"]      for n in names]
accs  = [multilayer_results[n]["accuracy"] for n in names]
x = np.arange(len(names)); w = 0.35
b1 = ax.bar(x-w/2, aucs, w, label="AUC-ROC",  color="steelblue", alpha=0.85, edgecolor="black")
b2 = ax.bar(x+w/2, accs, w, label="Accuracy", color="coral",     alpha=0.85, edgecolor="black")
for bar, v in zip(list(b1)+list(b2), aucs+accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{v:.2f}", ha="center", fontsize=8, fontweight="bold")
ax.axhline(0.8, color="orange", linestyle="--", linewidth=1.5)
ax.axhline(0.5, color="red",    linestyle="--", linewidth=1.5)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.set(ylabel="Score", ylim=(0, 1.15), title="B. Multi-Layer Probe Comparison")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis="y")

# C: Arms race
ax = fig.add_subplot(gs[1, :])
scenarios = list(arms_results.keys())
arm_aucs  = [arms_results[s]["auc"] for s in scenarios]
n_corrupt = [len(arms_results[s]["corrupted"]) for s in scenarios]
palette   = {0: "#2ecc71", 1: "#f39c12", 2: "#e67e22", 3: "#e74c3c"}
colors    = [palette[n] for n in n_corrupt]
bars = ax.bar(range(len(scenarios)), arm_aucs, color=colors, alpha=0.85, edgecolor="black")
for bar, v in zip(bars, arm_aucs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
ax.axhline(0.8, color="orange", linestyle="--", linewidth=2, label="AUC=0.8")
ax.axhline(0.5, color="red",    linestyle="--", linewidth=2, label="Chance")
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#2ecc71", label="No attack"),
    Patch(facecolor="#f39c12", label="1 layer corrupted"),
    Patch(facecolor="#e67e22", label="2 layers corrupted"),
    Patch(facecolor="#e74c3c", label="All layers corrupted"),
], fontsize=10, loc="lower right")
ax.set_xticks(range(len(scenarios))); ax.set_xticklabels(scenarios, fontsize=10)
ax.set(ylabel="AUC-ROC", ylim=(0, 1.15),
       title=f"C. Arms Race — Monitor: Layers {monitor}\nFull-coverage attack fails to degrade detection")
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle(f"GPT-2 Small | TruthfulQA | Best AUC: {probe_results['best_auc']:.3f}",
             fontsize=14, fontweight="bold")
plt.savefig(f"{CONFIG['output_dir']}/summary_figure.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")

## Step 11 · Save All Results

In [ ]:
import json, os

# Save probe results (already saved above, this adds arms race)
arms_data = {
    "monitor_mid_6_8_10":    {k: {"accuracy": v["accuracy"], "auc": v["auc"]} for k, v in arms_results.items()},
    "monitor_late_10_11_12": {k: {"accuracy": v["accuracy"], "auc": v["auc"]} for k, v in arms_late.items()},
}
with open(f"{CONFIG['output_dir']}/arms_race.json", "w") as f:
    json.dump(arms_data, f, indent=2)

with open(f"{CONFIG['output_dir']}/splits.json", "w") as f:
    json.dump({"train": train_data, "test": test_data, "seed": CONFIG["seed"]}, f, indent=2)

print("Outputs saved:")
for fname in os.listdir(CONFIG["output_dir"]):
    fpath = os.path.join(CONFIG["output_dir"], fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {fname:<40} {size/1024:.1f} KB")

print("\nTo download all outputs:")
print("  from google.colab import files")
print("  import shutil")
print("  shutil.make_archive('outputs', 'zip', './outputs')")
print("  files.download('outputs.zip')")